# Tenuto — Expressive Score-to-Performance AI Engine

Predicts human performance nuance (rubato, micro-timing, velocity, articulation, sustain pedal) from sheet music or MIDI scores.

### Stage 1: Setup Repository & Environment
**Note:** To enable GPU acceleration in Colab, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# 1. Clone repo if needed or pull latest changes
import os
if not os.path.exists("/content/tenuto"):
    !git clone https://github.com/kyleconciso/tenuto.git /content/tenuto
else:
    !cd /content/tenuto && git pull

# 2. Set working directory & PYTHONPATH globally
%cd /content/tenuto
%env PYTHONPATH=/content/tenuto:.

# 3. Verify PyTorch GPU & install dependencies
import torch
print("
Current Directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU. Switch to GPU in Colab via: Runtime > Change runtime type > T4 GPU")

!pip install -q partitura mido scipy tqdm matplotlib huggingface_hub midi2audio pandas pyarrow rclone


### Stage 2: Download Dataset

In [ ]:
# (Optional) Self-Hosted Storage Sync
# If you are hosting dataset/checkpoints on your PC using `python3 scripts/host_storage.py`:
# Set your public WebDAV URL below (e.g. https://xxx.lhr.life)
import os
%env HOST_STORAGE_URL=

host_url = os.environ.get("HOST_STORAGE_URL", "").strip()
if host_url:
    from src.sync import download_dataset_from_host
    download_dataset_from_host(host_url, target_dir="/content/tenuto/data/processed")


In [ ]:
%cd /content/tenuto
!PYTHONPATH=/content/tenuto python3 -m src.download_dataset --dataset combined --pianocore_subset PianoCoRe-A*

### Stage 3: Preprocess Dataset (40D Note Feature Tensors, Default Cap: 10,000 Pairs)

In [ ]:
%cd /content/tenuto
!PYTHONPATH=/content/tenuto python3 -m src.preprocess --data_dir ./data --processed_dir ./data/processed --max_samples 10000

### Stage 4: Train Transformer Backbone (Auto-Saves Checkpoints to Storage Host)

In [ ]:
%cd /content/tenuto
import os
host_url = os.environ.get("HOST_STORAGE_URL", "").strip()
cmd = f"!PYTHONPATH=/content/tenuto python3 -m src.train --model_type transformer --in_features 40 --epochs 20 --batch_size 16 --lr 0.0001"
if host_url:
    cmd += f" --host_url {host_url}"
get_ipython().system(cmd[1:])


### Stage 5: Expressive Inference

In [ ]:
%cd /content/tenuto
!PYTHONPATH=/content/tenuto python3 -m src.infer --score data/asap/Balakirev/Islamey/xml_score.musicxml --checkpoint checkpoints/best_transformer_model.pth --model_type transformer --output_midi output_expressive.mid

### Stage 6: Listenable Audio Comparison 🎧

In [ ]:
%cd /content/tenuto
!apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm timidity

import sys
sys.path.insert(0, "/content/tenuto")
from src.audio import play_audio_in_colab

print("🎵 1. Playing Original Flat Score (Mechanical):")
play_audio_in_colab("data/asap/Balakirev/Islamey/midi_score.mid", title="Original Flat Score")

print("\n🎵 2. Playing Original Human Performance (Ground Truth):")
play_audio_in_colab("data/asap/Balakirev/Islamey/CHEN04.mid", title="Human Performance")

print("\n🎵 3. Playing Tenuto AI Generated Performance:")
play_audio_in_colab("output_expressive.mid", title="Tenuto AI Expressive Performance")
